# Prepare environment

In [1]:
%load_ext autoreload
%autoreload 2

## Init base libs

* Install libs

In [2]:
%%capture
%pip install \
    python-dotenv

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
print(os.getenv('DB_DRIVER', 'postgresql'))

postgresql


## Debezium

Manage the Debezium Postgres connector through the Kafka Connect REST API, using the
definition kept in `../debezium-connector/postgres-connector.json` (the same file as
`scripts/connector.sh register`). It's mounted read-only at `/debezium-connector` in the
jupyter container (see `docker-compose.yml`).

Captured changes go to Kafka topics `<topic.prefix>.<schema>.<table>`, for example
`ecommerce_db.inventory.customers`, read by the Flink job `flink-cdc-es`.

In [4]:
%%capture
%pip install \
   requests

In [5]:
import json
import os
import requests

# Kafka Connect REST API, as seen from the jupyter container (service `kafka-connect`)
CONNECT_URL = os.getenv("KAFKA_CONNECT_URL", "http://kafka-connect:8083")
# Connector definition shared with scripts/connector.sh (mounted from ../debezium-connector)
CONNECTOR_FILE = os.getenv("CONNECTOR_FILE", "/debezium-connector/postgres-connector.json")


def show(response):
    """Print the request, the status code and the JSON (or text) body of a response."""
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=4))
    except ValueError:
        print(response.text)

### List connectors

`expand=status` returns the state of each connector and of its tasks in one call.

In [6]:
show(requests.get(f"{CONNECT_URL}/connectors", params={"expand": "status"}))

GET http://kafka-connect:8083/connectors?expand=status -> 200
{
    "postgres-connector": {
        "status": {
            "name": "postgres-connector",
            "connector": {
                "state": "RUNNING",
                "worker_id": "172.22.0.11:8083"
            },
            "tasks": [
                {
                    "id": 0,
                    "state": "RUNNING",
                    "worker_id": "172.22.0.11:8083"
                }
            ],
            "type": "source"
        }
    }
}


### Load the connector definition

Read the JSON file and print each property with what it does. JSON has no comments,
so the explanations live in `PROPERTY_COMMENTS`. Passwords are masked in the output.

In [7]:
with open(CONNECTOR_FILE) as f:
    connector = json.load(f)

PROPERTY_COMMENTS = {
    "connector.class": "Debezium source connector for Postgres",
    "database.hostname": "Postgres service on the compose network (hostname `database`)",
    "database.port": "Postgres port inside the network (not the host port)",
    "database.user": "user Debezium connects with (needs the REPLICATION privilege)",
    "database.password": "password of database.user",
    "database.dbname": "database to capture",
    "topic.prefix": "topic name = <topic.prefix>.<schema>.<table>",
    "schema.include.list": "capture only the tables of these schemas",
    "plugin.name": "logical decoding plugin: pgoutput is built into Postgres 10+",
    "slot.name": "replication slot keeping the WAL position, one per connector",
    "publication.name": "Postgres publication listing the captured tables",
    "publication.autocreate.mode": "filtered: create the publication for the included tables only",
    "key.converter": "message key (the row primary key) serialized as JSON",
    "key.converter.schemas.enable": "false: plain JSON, no schema/payload wrapper",
    "value.converter": "message value (Debezium envelope before/after/source/op/ts_ms) as JSON",
    "value.converter.schemas.enable": "false: plain JSON, the format flink-cdc-es expects",
}

print(f"name: {connector['name']}  ({CONNECTOR_FILE})")
width = max(len(k) for k in connector["config"])
value_width = max(len(str(v)) for v in connector["config"].values())
for key, value in connector["config"].items():
    shown = "****" if "password" in key else value
    print(f"  {key:<{width}} = {shown:<{value_width}} # {PROPERTY_COMMENTS.get(key, '')}")

name: postgres-connector  (/debezium-connector/postgres-connector.json)
  connector.class                = io.debezium.connector.postgresql.PostgresConnector # Debezium source connector for Postgres
  database.hostname              = database                                           # Postgres service on the compose network (hostname `database`)
  database.port                  = 5432                                               # Postgres port inside the network (not the host port)
  database.user                  = debezium                                           # user Debezium connects with (needs the REPLICATION privilege)
  database.password              = ****                                               # password of database.user
  database.dbname                = ecommerce                                          # database to capture
  topic.prefix                   = ecommerce_db                                       # topic name = <topic.prefix>.<schema>.<table>
  sch

### Register or update the connector

`PUT /connectors/<name>/config` creates the connector when it doesn't exist and updates it
otherwise. When the live config already matches the file, nothing is sent, so a running
connector isn't restarted.

In [8]:
name, config = connector["name"], connector["config"]

current = requests.get(f"{CONNECT_URL}/connectors/{name}/config")
live_config = {k: v for k, v in current.json().items() if k != "name"} if current.ok else None

if live_config == config:
    print(f"'{name}' is already registered with the same config: nothing to do")
else:
    print(f"'{name}' is " + ("registered with another config: updating" if current.ok else "not registered: creating"))
    show(requests.put(f"{CONNECT_URL}/connectors/{name}/config", json=config))

'postgres-connector' is already registered with the same config: nothing to do


### Check the status

The connector and every task must be `RUNNING`. A `FAILED` task has a `trace` field with
the cause.

In [9]:
response = requests.get(f"{CONNECT_URL}/connectors/{name}/status")
status = response.json()

print(f"connector {name}: {status['connector']['state']}")
for task in status["tasks"]:
    print(f"  task {task['id']}: {task['state']}")
    if "trace" in task:
        print(task["trace"])

connector postgres-connector: RUNNING
  task 0: RUNNING


### Topics written by the connector

One topic per captured table, listed once the table had at least one change (or snapshot).

In [10]:
response = requests.get(f"{CONNECT_URL}/connectors/{name}/topics")
for topic in response.json()[name]["topics"]:
    print(topic)

ecommerce_db.inventory.phone_numbers
ecommerce_db.inventory.customers
